# Transformación de Datos: Bronze → Silver → Gold
## Desafío 4 - Del Dato a la Acción Hackathon

Este notebook implementa el patrón Medallion para transformación de datos en Microsoft Fabric.

**Pre-requisitos:**
- Agregar los tres Lakehouses como recursos del notebook: `Lakehouse_Bronze`, `Lakehouse_Silver`, `Lakehouse_Gold`
- Asegurar que `Lakehouse_Bronze` es el Lakehouse por defecto

---
## Bronze → Silver
### Lectura de datos crudos desde Lakehouse Bronze y limpieza de datos

In [ ]:
# Leer la tabla cruda desde Lakehouse Bronze
# Alternativa con ruta completa:
# df_orders_bronze = spark.read.format("delta").load(
#     "abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_bronze_id>/Tables/sqls_salesorderheader"
# )

# Alternativa más sencilla si Bronze es el Lakehouse default:
df_orders_bronze = spark.table("sqls_salesorderheader")

# Ver el esquema y primeras filas
df_orders_bronze.printSchema()
df_orders_bronze.show(5)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# 1. Eliminar duplicados
df_orders_silver = df_orders_bronze.dropDuplicates(["SalesOrderID"])

# 2. Tratar valores nulos
df_orders_silver = df_orders_silver.fillna({
    "Comment": "Sin comentario",
    "PurchaseOrderNumber": "N/A"
})

# 3. Estandarizar tipos de datos: convertir fechas
df_orders_silver = df_orders_silver.withColumn(
    "OrderDate", F.to_date(F.col("OrderDate"))
).withColumn(
    "DueDate", F.to_date(F.col("DueDate"))
).withColumn(
    "ShipDate", F.to_date(F.col("ShipDate"))
)

# 4. Estandarizar textos: mayúsculas en campos de texto clave
df_orders_silver = df_orders_silver.withColumn(
    "Status_std", F.when(F.col("Status") == 5, "Enviado")
                   .when(F.col("Status") == 4, "En proceso")
                   .otherwise("Otro")
)

# 5. Agregar columna de auditoría
df_orders_silver = df_orders_silver.withColumn(
    "_silver_ingestion_ts", F.current_timestamp()
)

df_orders_silver.show(5)

In [ ]:
# Escribir como Delta Table en Lakehouse Silver
df_orders_silver.write.format("delta").mode("overwrite").saveAsTable(
    "Lakehouse_Silver.salesorderheader"
)
print("✅ Tabla salesorderheader escrita en Lakehouse Silver")

### Procesamiento de tabla Customer

In [ ]:
# Leer tabla Customer desde Bronze
df_customer_bronze = spark.table("sqls_customer")

# Limpieza de emails nulos, estandarización de nombres
df_customer_silver = df_customer_bronze.dropDuplicates(["CustomerID"])

df_customer_silver = df_customer_silver.fillna({
    "EmailAddress": "sin_email@unknown.com",
    "CompanyName": "Sin empresa",
    "FirstName": "Desconocido",
    "LastName": "Desconocido"
})

# Estandarizar nombres (capitalizar)
df_customer_silver = df_customer_silver.withColumn(
    "FirstName", F.initcap(F.col("FirstName"))
).withColumn(
    "LastName", F.initcap(F.col("LastName"))
)

# Agregar columna de auditoría
df_customer_silver = df_customer_silver.withColumn(
    "_silver_ingestion_ts", F.current_timestamp()
)

# Escribir en Lakehouse Silver
df_customer_silver.write.format("delta").mode("overwrite").saveAsTable(
    "Lakehouse_Silver.customer"
)
print("✅ Tabla customer escrita en Lakehouse Silver")

### Procesamiento de tabla SalesOrderDetail

In [ ]:
# Leer tabla SalesOrderDetail desde Bronze
df_detail_bronze = spark.table("sqls_salesorderdetail")

# Eliminar duplicados y validar cantidades
df_detail_silver = df_detail_bronze.dropDuplicates(["SalesOrderDetailID"])

# Validar que las cantidades sean positivas
df_detail_silver = df_detail_silver.filter(F.col("OrderQty") > 0)

# Agregar columna de auditoría
df_detail_silver = df_detail_silver.withColumn(
    "_silver_ingestion_ts", F.current_timestamp()
)

# Escribir en Lakehouse Silver
df_detail_silver.write.format("delta").mode("overwrite").saveAsTable(
    "Lakehouse_Silver.salesorderdetail"
)
print("✅ Tabla salesorderdetail escrita en Lakehouse Silver")

### Procesamiento de tabla Product

In [ ]:
# Leer tabla Product desde Bronze
df_product_bronze = spark.table("sqls_product")

# Estandarización de nombres de producto
df_product_silver = df_product_bronze.dropDuplicates(["ProductID"])

# Tratar valores nulos
df_product_silver = df_product_silver.fillna({
    "Color": "Sin color",
    "Size": "N/A",
    "Weight": 0
})

# Agregar columna de auditoría
df_product_silver = df_product_silver.withColumn(
    "_silver_ingestion_ts", F.current_timestamp()
)

# Escribir en Lakehouse Silver
df_product_silver.write.format("delta").mode("overwrite").saveAsTable(
    "Lakehouse_Silver.product"
)
print("✅ Tabla product escrita en Lakehouse Silver")

---
## Silver → Gold
### Agregaciones y modelado dimensional para consumo analítico

In [ ]:
# Leer tablas Silver
df_orders = spark.table("Lakehouse_Silver.salesorderheader")
df_details = spark.table("Lakehouse_Silver.salesorderdetail")
df_customers = spark.table("Lakehouse_Silver.customer")
df_products = spark.table("Lakehouse_Silver.product")

# Crear fact_ventas: unión de órdenes y detalles
fact_ventas = df_orders.join(df_details, "SalesOrderID", "inner") \
    .select(
        F.col("SalesOrderID").alias("id_orden"),
        F.col("SalesOrderDetailID").alias("id_detalle"),
        F.col("CustomerID").alias("id_cliente"),
        F.col("ProductID").alias("id_producto"),
        F.col("OrderDate").alias("fecha_orden"),
        F.col("OrderQty").cast("integer").alias("cantidad"),
        F.col("UnitPrice").cast("double").alias("precio_unitario"),
        F.col("LineTotal").cast("double").alias("monto_total")
    )

fact_ventas.show(5)

In [ ]:
# Crear dimensión de clientes
dim_cliente = df_customers.select(
    F.col("CustomerID").alias("id_cliente"),
    F.concat_ws(" ", F.col("FirstName"), F.col("LastName")).alias("nombre_cliente"),
    F.col("EmailAddress").alias("email"),
    F.col("CompanyName").alias("empresa")
).distinct()

dim_cliente.show(5)

In [ ]:
# Crear dimensión de productos
dim_producto = df_products.select(
    F.col("ProductID").alias("id_producto"),
    F.col("Name").alias("nombre_producto"),
    F.col("ProductNumber").alias("codigo_producto"),
    F.col("StandardCost").cast("double").alias("costo_estandar"),
    F.col("ListPrice").cast("double").alias("precio_lista"),
    F.col("Color").alias("color")
).distinct()

dim_producto.show(5)

In [ ]:
from pyspark.sql.types import StructType, StructField, DateType, IntegerType, StringType
import pandas as pd

# Generar tabla de fechas a partir del rango de datos
fecha_min = fact_ventas.agg(F.min("fecha_orden")).collect()[0][0]
fecha_max = fact_ventas.agg(F.max("fecha_orden")).collect()[0][0]

fechas = pd.date_range(start=fecha_min, end=fecha_max, freq='D')
df_fechas = pd.DataFrame({
    "fecha": fechas,
    "anio": fechas.year,
    "mes": fechas.month,
    "dia": fechas.day,
    "nombre_mes": fechas.strftime('%B'),
    "trimestre": fechas.quarter,
    "semana_anio": fechas.isocalendar().week.astype(int)
})

dim_fecha = spark.createDataFrame(df_fechas)
dim_fecha.show(5)

In [ ]:
# Escribir todas las tablas en Lakehouse Gold
fact_ventas.write.format("delta").mode("overwrite").saveAsTable("Lakehouse_Gold.fact_ventas")
dim_cliente.write.format("delta").mode("overwrite").saveAsTable("Lakehouse_Gold.dim_cliente")
dim_producto.write.format("delta").mode("overwrite").saveAsTable("Lakehouse_Gold.dim_producto")
dim_fecha.write.format("delta").mode("overwrite").saveAsTable("Lakehouse_Gold.dim_fecha")

print("✅ Tablas Gold creadas exitosamente:")
print("   - fact_ventas")
print("   - dim_cliente")
print("   - dim_producto")
print("   - dim_fecha")

---
## Validación Final
### Verificar que las tablas se crearon correctamente

In [ ]:
# Contar registros en cada tabla Gold
print("Conteo de registros en Lakehouse Gold:")
print(f"fact_ventas: {spark.table('Lakehouse_Gold.fact_ventas').count()}")
print(f"dim_cliente: {spark.table('Lakehouse_Gold.dim_cliente').count()}")
print(f"dim_producto: {spark.table('Lakehouse_Gold.dim_producto').count()}")
print(f"dim_fecha: {spark.table('Lakehouse_Gold.dim_fecha').count()}")